# **MÉTODOS DE REPRESENTACIÓN TRADICIONALES**

## **TF-IDF**

In [31]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

In [32]:
data = pd.read_parquet("pruebasData/preprocNoContext.parquet")
data = data[["topic","ticker","headline","article_text","text_nc"]]

print(f"\nDocumentos: {len(data)}")
data.head(5)


Documentos: 5160


,topic,ticker,headline,article_text,text_nc
0,2_company_year_u_ha,AAPL,UK pay growth slows and unemployment ticks hig...,The UK jobs market continues to show signs of ...,uk job market continue sign weakness pay growt...
1,1_think_going_just_like,AAPL,Rising pension age hits women hardest,New data from the Department from Work and Pen...,new datum department work pension work pattern...
2,1_think_going_just_like,AAPL,How to ask for changes at work if you are neur...,Asking for workplace accommodations is often e...,ask workplace accommodation easy people worry ...
3,2_company_year_u_ha,AAPL,Apple announces major expansion of renewables ...,Apple has announced a major expansion of its r...,apple announce major expansion renewable inves...
4,0_ai_company_nvidia_billion,AAPL,"1 Unstoppable Stock Poised to Join Nvidia, App...",The advent of artificial intelligence (AI) les...,advent artificial intelligence ai year ago spa...


### Crear el vectorizer TF-IDF

In [ ]:
# Crear el vectorizador TF-IDF
vectorizer = TfidfVectorizer(
    min_df=2,           # Ignorar términos que aparecen en menos de 2 documentos
    max_df=0.8,         # Ignorar términos que aparecen en más del 80% de los documentos
    ngram_range=(1, 2), # Usar unigramas y bigramas
    max_features=5000   # Limitar a las 5000 características más importantes
)

In [34]:
corpus = data["text_nc"].tolist()
tfidf_matrix = vectorizer.fit_transform(corpus)

print(f"\nForma de la matriz TF-IDF: {tfidf_matrix.shape}")
print(f"  - Documentos: {tfidf_matrix.shape[0]}")
print(f"  - Características (términos únicos): {tfidf_matrix.shape[1]}")



Forma de la matriz TF-IDF: (5160, 5000)
  - Documentos: 5160
  - Características (términos únicos): 5000


### ANÁLISIS DE PALABRAS MÁS RELEVANTES

In [35]:
feature_names = vectorizer.get_feature_names_out()

def get_top_words(tfidf_vector, feature_names, top_n=10):
    """Retorna las top_n palabras con el mayor valor TF-IDF."""
    # Convertir a array denso si es disperso
    if hasattr(tfidf_vector, 'toarray'):
        tfidf_vector = tfidf_vector.toarray().flatten()
    
    sorted_indices = np.argsort(tfidf_vector)[-top_n:][::-1]
    return [(feature_names[i], tfidf_vector[i]) for i in sorted_indices if tfidf_vector[i] > 0]

In [36]:
for idx in range(min(5, len(corpus))):
    print(f"\nDocumento {idx}:")
    print(f"Texto original (primeros 100 chars): {data.iloc[idx]['article_text'][:100]}...")
    print(f"\nTop 10 palabras más relevantes:")
    top_words = get_top_words(tfidf_matrix[idx], feature_names, top_n=10)
    for word, score in top_words:
        print(f"  {word:<30} {score:.4f}")
    print("-" * 60)


Documento 0:
Texto original (primeros 100 chars): The UK jobs market continues to show signs of weakness, with pay growth slowing and unemployment edg...

Top 10 palabras más relevantes:
  number                         0.2598
  budget                         0.2198
  payroll                        0.1979
  fall                           0.1977
  july                           0.1965
  statistic                      0.1961
  month                          0.1948
  fall num                       0.1935
  num                            0.1903
  august                         0.1833
------------------------------------------------------------

Documento 1:
Texto original (primeros 100 chars): New data from the Department from Work and Pensions on the working patterns of people aged 50 and ab...

Top 10 palabras más relevantes:
  pension                        0.6762
  work                           0.3412
  age                            0.3117
  people                         0.2035
  l

### SIMILITUD ENTRE DOCS

In [37]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

# Calcula similitud entre los primeros 5 documentos
n_docs_to_compare = min(5, len(corpus))
similarities = cosine_similarity(tfidf_matrix[:n_docs_to_compare])

# Crear DataFrame para visualización
sim_matrix_df = pd.DataFrame(
    similarities,
    index=[f"Doc {i}: {data['headline'].iloc[i][:40]}..." for i in range(n_docs_to_compare)],
    columns=[f"Doc {j}" for j in range(n_docs_to_compare)]
)

print("\nMatriz de similitud coseno entre los primeros documentos:\n")
print(sim_matrix_df.round(3))



Matriz de similitud coseno entre los primeros documentos:

                                                    Doc 0  Doc 1  Doc 2  \
Doc 0: UK pay growth slows and unemployment tic...  1.000  0.080  0.070   
Doc 1: Rising pension age hits women hardest...     0.080  1.000  0.130   
Doc 2: How to ask for changes at work if you ar...  0.070  0.130  1.000   
Doc 3: Apple announces major expansion of renew...  0.068  0.028  0.065   
Doc 4: 1 Unstoppable Stock Poised to Join Nvidi...  0.064  0.030  0.041   

                                                    Doc 3  Doc 4  
Doc 0: UK pay growth slows and unemployment tic...  0.068  0.064  
Doc 1: Rising pension age hits women hardest...     0.028  0.030  
Doc 2: How to ask for changes at work if you ar...  0.065  0.041  
Doc 3: Apple announces major expansion of renew...  1.000  0.094  
Doc 4: 1 Unstoppable Stock Poised to Join Nvidi...  0.094  1.000  


### BÚSQUEDA POR QUERY

In [38]:
print("\n" + "="*60)
print("BÚSQUEDA POR QUERY")
print("="*60)

# Ejemplo de queries
queries = [
    "stock market growth",
    "unemployment rate",
    "artificial intelligence technology",
    "climate change policy"
]

for query in queries:
    print(f"\nQuery: '{query}'")
    print("-" * 40)
    
    # Transformar query usando el mismo vectorizador
    query_vec = vectorizer.transform([query])
    
    # Calcular similitud con todos los documentos
    similarities = cosine_similarity(query_vec, tfidf_matrix).flatten()
    
    # Obtener los 3 documentos más similares
    top_indices = np.argsort(similarities)[-3:][::-1]
    
    for rank, idx in enumerate(top_indices, 1):
        if similarities[idx] > 0:
            print(f"\n{rank}. Similitud: {similarities[idx]:.4f}")
            print(f"   Texto: {data.iloc[idx]['article_text'][:150]}...")


BÚSQUEDA POR QUERY

Query: 'stock market growth'
----------------------------------------

1. Similitud: 0.3447
   Texto: The Stock Market Is Suffering an AI Hangover. What to Do Now.
Tech stocks are tumbling, and that’s OK—as long as you remember that there’s more to th...

2. Similitud: 0.3260
   Texto: After a few decades as a daily stockpicking column, Questor returns to its 60-year-old roots by taking a weekly view of the markets – what is moving t...

3. Similitud: 0.3260
   Texto: After a few decades as a daily stockpicking column, Questor returns to its 60-year-old roots by taking a weekly view of the markets – what is moving t...

Query: 'unemployment rate'
----------------------------------------

1. Similitud: 0.4543
   Texto: The market has long anticipated the Federal Reserve's quarter-point rate cut, but as ever, the next question is what and when its next move will be. O...

2. Similitud: 0.3400
   Texto: By Howard Schneider
WASHINGTON (Reuters) -The Chicago Federal Res

### EMBEDDINGS

In [20]:

print("\n" + "="*60)
print("GUARDANDO EMBEDDINGS")
print("="*60)

# Convertir matriz dispersa a densa (cuidado con memoria si tienes muchos documentos)
tfidf_dense = tfidf_matrix.toarray()

# Crear DataFrame con los embeddings
embeddings_df = pd.DataFrame(
    tfidf_dense,
    columns=[f"tfidf_{i}" for i in range(tfidf_dense.shape[1])]
)

# Añadir información del documento original
embeddings_df['doc_index'] = data.index
embeddings_df['article_text'] = data['article_text'].values
embeddings_df['text_nc'] = data['text_nc'].values

# Guardar
embeddings_df.to_csv("pruebasData/tfidf_embeddings.csv", index=False)
print(f"\n✓ Embeddings guardados en 'pruebasData/tfidf_embeddings.csv'")
print(f"  Forma: {embeddings_df.shape}")

# También guardar el vectorizador para uso futuro
import pickle
with open("pruebasData/tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(vectorizer, f)
print(f"\n✓ Vectorizador guardado en 'pruebasData/tfidf_vectorizer.pkl'")

print("\n" + "="*60)
print("PROCESO COMPLETADO")
print("="*60)


GUARDANDO EMBEDDINGS

✓ Embeddings guardados en 'pruebasData/tfidf_embeddings.csv'
  Forma: (5160, 5003)

✓ Vectorizador guardado en 'pruebasData/tfidf_vectorizer.pkl'

PROCESO COMPLETADO


# PRUEBA DE VERDAD

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
# =========================================
print("\n" + "="*60)
print("CREANDO VECTORIZADOR TF-IDF")
print("="*60)

# Eliminar filas con texto vacío o NaN
data_clean = data[data["text_nc"].notna() & (data["text_nc"].str.len() > 0)].copy()
print(f"Documentos válidos después de limpieza: {len(data_clean)}")

# Crear el vectorizador TF-IDF
vectorizer = TfidfVectorizer(
    min_df=2,           # Ignorar términos que aparecen en menos de 2 documentos
    max_df=0.8,         # Ignorar términos que aparecen en más del 80% de los documentos
    ngram_range=(1, 2), # Usar unigramas y bigramas
    max_features=5000   # Limitar a las 5000 características más importantes
)

# Ajustar el vectorizador con los textos preprocesados
corpus = data_clean["text_nc"].tolist()
tfidf_matrix = vectorizer.fit_transform(corpus)

print(f"\nForma de la matriz TF-IDF: {tfidf_matrix.shape}")
print(f"  - Documentos: {tfidf_matrix.shape[0]}")
print(f"  - Características (términos únicos): {tfidf_matrix.shape[1]}")

# =========================================
# ANÁLISIS DE PALABRAS MÁS RELEVANTES
# =========================================
print("\n" + "="*60)
print("PALABRAS MÁS RELEVANTES POR DOCUMENTO")
print("="*60)

# Obtener nombres de características
feature_names = vectorizer.get_feature_names_out()

def get_top_words(tfidf_vector, feature_names, top_n=10):
    """Retorna las top_n palabras con el mayor valor TF-IDF."""
    # Convertir a array denso si es disperso
    if hasattr(tfidf_vector, 'toarray'):
        tfidf_vector = tfidf_vector.toarray().flatten()
    
    sorted_indices = np.argsort(tfidf_vector)[-top_n:][::-1]
    return [(feature_names[i], tfidf_vector[i]) for i in sorted_indices if tfidf_vector[i] > 0]

# Mostrar las palabras más relevantes de los primeros 5 documentos
for idx in range(min(5, len(corpus))):
    print(f"\nDocumento {idx}:")
    print(f"Texto original (primeros 100 chars): {data_clean.iloc[idx]['article_text'][:100]}...")
    print(f"\nTop 10 palabras más relevantes:")
    top_words = get_top_words(tfidf_matrix[idx], feature_names, top_n=10)
    for word, score in top_words:
        print(f"  {word:<30} {score:.4f}")
    print("-" * 60)

# =========================================
# SIMILITUD ENTRE DOCUMENTOS
# =========================================
print("\n" + "="*60)
print("SIMILITUD COSENO ENTRE DOCUMENTOS")
print("="*60)

# Calcular similitud entre los primeros 5 documentos
n_docs_to_compare = min(5, len(corpus))
similarities = cosine_similarity(tfidf_matrix[:n_docs_to_compare])

print(f"\nMatriz de similitud entre los primeros {n_docs_to_compare} documentos:\n")
for i in range(n_docs_to_compare):
    for j in range(i+1, n_docs_to_compare):
        print(f"Doc {i} vs Doc {j}: {similarities[i][j]:.4f}")

# =========================================
# BÚSQUEDA POR QUERY
# =========================================
print("\n" + "="*60)
print("BÚSQUEDA POR QUERY")
print("="*60)

# Ejemplo de queries
queries = [
    "stock market growth",
    "unemployment rate",
    "artificial intelligence technology",
    "climate change policy"
]

for query in queries:
    print(f"\nQuery: '{query}'")
    print("-" * 40)
    
    # Transformar query usando el mismo vectorizador
    query_vec = vectorizer.transform([query])
    
    # Calcular similitud con todos los documentos
    similarities = cosine_similarity(query_vec, tfidf_matrix).flatten()
    
    # Obtener los 3 documentos más similares
    top_indices = np.argsort(similarities)[-3:][::-1]
    
    for rank, idx in enumerate(top_indices, 1):
        if similarities[idx] > 0:
            print(f"\n{rank}. Similitud: {similarities[idx]:.4f}")
            print(f"   Texto: {data_clean.iloc[idx]['article_text'][:150]}...")

# =========================================
# GUARDAR EMBEDDINGS
# =========================================
print("\n" + "="*60)
print("GUARDANDO EMBEDDINGS")
print("="*60)

# Convertir matriz dispersa a densa (cuidado con memoria si tienes muchos documentos)
tfidf_dense = tfidf_matrix.toarray()

# Crear DataFrame con los embeddings
embeddings_df = pd.DataFrame(
    tfidf_dense,
    columns=[f"tfidf_{i}" for i in range(tfidf_dense.shape[1])]
)

# Añadir información del documento original
embeddings_df['doc_index'] = data_clean.index
embeddings_df['article_text'] = data_clean['article_text'].values
embeddings_df['text_nc'] = data_clean['text_nc'].values

# Guardar
embeddings_df.to_csv("pruebasData/tfidf_embeddings.csv", index=False)
print(f"\n✓ Embeddings guardados en 'pruebasData/tfidf_embeddings.csv'")
print(f"  Forma: {embeddings_df.shape}")

# También guardar el vectorizador para uso futuro
import pickle
with open("pruebasData/tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(vectorizer, f)
print(f"\n✓ Vectorizador guardado en 'pruebasData/tfidf_vectorizer.pkl'")

print("\n" + "="*60)
print("PROCESO COMPLETADO")
print("="*60)

### PRUEBA MONTAR EMBEDDING

## EVALUACIÓN TRADICIONAL

In [32]:
# === Evaluación exploratoria del embedding TF-IDF ===
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

n_docs, n_features = X.shape
sparsity = X.nnz / (n_docs * n_features)  # proporción de elementos distintos de 0
print(f"📘 Shape TF-IDF: {X.shape} (docs x vocab)")
print(f"📊 Sparsity: {sparsity:.6f}  ({sparsity*100:.4f}% de celdas no nulas)")
print(f"🔠 Tamaño del vocabulario: {len(vec.vocabulary_):,}")

# --- 2️⃣ Términos más comunes / más raros ---
terms = np.array(vec.get_feature_names_out())
idf = vec.idf_

common_terms = terms[np.argsort(idf)[:10]]
rare_terms   = terms[np.argsort(-idf)[:10]]

print("\n🔥 10 términos más comunes (bajo IDF):")
print(", ".join(common_terms))
print("\n🧊 10 términos más raros (alto IDF):")
print(", ".join(rare_terms))

# --- 3️⃣ Top términos de un documento específico ---
def top_terms_for_doc(i, vec, X, n=10):
    row = X[i].toarray().ravel()
    top_idx = row.argsort()[::-1][:n]
    return list(zip(terms[top_idx], row[top_idx]))

idx_example = 0  # cambia el índice para otros documentos
top_terms = top_terms_for_doc(idx_example, vec, X, n=10)
print(f"\n📰 Top 10 términos del documento {idx_example}:")
for term, weight in top_terms:
    print(f"{term:<20} {weight:.5f}")

# --- 4️⃣ Similitud entre documentos ---
if X.shape[0] > 1:
    sim = cosine_similarity(X[0], X[1])[0,0]
    print(f"\n🤝 Similitud coseno entre documento 0 y 1: {sim:.4f}")

# --- 5️⃣ (Opcional) Guardar resumen rápido ---
with open(SAVE_DIR / "embedding_summary.txt", "w", encoding="utf-8") as f:
    f.write(f"Shape: {X.shape}\n")
    f.write(f"Sparsity: {sparsity:.6f}\n")
    f.write("Top comunes: " + ", ".join(common_terms) + "\n")
    f.write("Top raros: " + ", ".join(rare_terms) + "\n")
    f.write(f"Ejemplo doc {idx_example}: " +
            ", ".join([t for t,_ in top_terms]) + "\n")
print("\n✅ Informe de embedding guardado en:", SAVE_DIR / "embedding_summary.txt")


📘 Shape TF-IDF: (5160, 104149) (docs x vocab)
📊 Sparsity: 0.002787  (0.2787% de celdas no nulas)
🔠 Tamaño del vocabulario: 104,149

🔥 10 términos más comunes (bajo IDF):
year, company, num, ai, __percent__, market, stock, new, share, high

🧊 10 términos más raros (alto IDF):
zuckerberg year, zuckerberg widely, zuckerberg wednesday, zuckerberg time, zuckerberg talk, gpu operate, zuckerberg recognition, zuckerberg recently, zuckerberg quickly, zuckerberg pull

📰 Top 10 términos del documento 0:
autumn budget        0.19996
ons                  0.19609
autumn               0.18688
unemployment         0.17604
fall num             0.17263
estimate number      0.15747
number               0.15146
num july             0.14965
vacancy              0.14668
continue sign        0.14668

🤝 Similitud coseno entre documento 0 y 1: 0.0351

✅ Informe de embedding guardado en: embeddings_tfidf\embedding_summary.txt
